In [1]:
# !pip install --upgrade transformers datasets evaluate

In [2]:
import re
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    matthews_corrcoef,
    balanced_accuracy_score,
)
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from datasets import (
    load_dataset, 
    ClassLabel, 
)
import evaluate

/usr/local/lib/python3.10/dist-packages/torchvision/datapoints/__init__.py:14: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/usr/local/lib/python3.10/dist-packages/torchvision/transforms/v2/__init__.py:64: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https:/

In [38]:
MODEL_NAME = "distilbert-base-uncased"

TEST_SIZE = 0.2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 2
BATCH_SIZE = 16

DATA_PATH = "data/sentence_sets_trimmed.csv"
DATA_ENCODING = "ISO-8859-1"

LABEL_COLUMN = "applicant_gender"
TEXT_COLUMN = "s1_s2"

STRATIFY_ENABLED = True
DEGENDER_ENABLED = True
BALANCED_WEIGHTS_ENABLED = False

DEGENDER_LEVEL = "original"
BALANCED_TYPE = "oversample"

MAX_LENGTH = 512

OUTPUT_NAME = f"{MODEL_NAME}-finetuned-nlp-letters-{TEXT_COLUMN}"
OUTPUT_NAME += "-degendered" if DEGENDER_ENABLED else ""
OUTPUT_NAME += "-stratified" if STRATIFY_ENABLED else ""
OUTPUT_NAME += "-balanced" if BALANCED_WEIGHTS_ENABLED else ""

In [39]:
if torch.cuda.is_available():
    print("CUDA is available. Using GPU:", torch.cuda.get_device_name(0))
    print("Number of GPUs available:", torch.cuda.device_count())
else:
    print("CUDA is not available. Running on CPU.")

CUDA is available. Using GPU: Tesla V100-PCIE-16GB
Number of GPUs available: 1


In [40]:
class LettersBERTModule(nn.Module):
    def __init__(self, model_name, num_labels, class_weights=None, cls_or_mean="cls"):
        super().__init__()

        self.config = AutoConfig.from_pretrained(model_name)
        self.config.num_labels = num_labels
        self.config.class_weights = class_weights
        self.config.cls_or_mean = cls_or_mean

        # Layers
        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        self.classifier = nn.Linear(self.config.hidden_size, self.config.num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)

        # Either use [CLS] token or mean pooling
        if self.config.cls_or_mean == "mean":
            output = torch.mean(outputs.last_hidden_state, dim=1)
        else:
            output = outputs.last_hidden_state[:, 0, :]

        # Logits and loss
        logits = self.classifier(output)
        loss = None

        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.config.class_weights)
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}

In [41]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [42]:
def degender(text, level="original"):
    if level == "enhanced":
        titles = r"\b(?:mr|mrs|ms|miss|mister|sir|madam)\b"
        nouns = r"\b(?:man|men|woman|women|gentleman|lady|boy|boys|girl|girls)(?:'s)?\b"
        pronouns = r"\b(?:he|him|his|she|her|hers|himself|herself)\b"
    else:
        titles = r"\b(?:mr|mrs|ms|miss|mister)\b"
        nouns = r"\b(?:man|men|woman|women|gentleman|lady)(?:'s)?\b"
        pronouns = r"\b(?:he|him|his|she|her|hers)\b"

    titles_regex = re.compile(titles, flags=re.IGNORECASE)
    nouns_regex = re.compile(nouns, flags=re.IGNORECASE)
    pronouns_regex = re.compile(pronouns, flags=re.IGNORECASE)

    text = titles_regex.sub("mx", text)
    text = nouns_regex.sub("person", text)
    text = pronouns_regex.sub("they", text)

    return text

In [43]:
def preprocess(data):
    texts = data[TEXT_COLUMN]

    if DEGENDER_ENABLED:
        texts = [degender(t, DEGENDER_LEVEL) for t in texts]

    tokenized = tokenizer(
        texts, 
        truncation=True, 
        padding=True
    )

    tokenized["labels"] = data[LABEL_COLUMN]

    return tokenized

In [44]:
# Load the data
dataset = load_dataset("csv", data_files=DATA_PATH, encoding=DATA_ENCODING)

# Recast labels as class features
unique_labels = dataset["train"].unique(LABEL_COLUMN)

features = dataset["train"].features
features[LABEL_COLUMN] = ClassLabel(names=unique_labels)

dataset = dataset.cast(features)

dataset = dataset["train"].train_test_split(
    test_size=TEST_SIZE,
    stratify_by_column=LABEL_COLUMN if STRATIFY_ENABLED else None,
    seed=100,
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [45]:
print([degender(t) for t in dataset["train"][TEXT_COLUMN][:10]])

["FIRST_NAME MIDDLE_NAME paz de araujo   a fourth   year medical student at the university of colorado health sciences center   currently applying for a position in your anesthesiology residency program  * I had the opportunity to work with FIRST_NAME during clinical clerkships at children's hospital of colorado in both they third and fourth year of medical school  * FIRST_NAME excelled in all phases of they clinical responsibilities during they pediatric anesthesiology clerkship  * I often found FIRST_NAME remaining in the or later than usual to observe and participate in interesting cases taking place later in the FIRST_NAME  * airway and intravenous access skills were outstanding and significantly above the standard expected for FIRST_NAME's current level of training  * FIRST_NAME displayed a comfort and keen ability to interact and connect with pediatric patients and their families  * FIRST_NAME's commitment to public health and research are evident with they participation in the s

In [46]:
train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

Map:   0%|          | 0/2628 [00:00<?, ? examples/s]

Map:   0%|          | 0/657 [00:00<?, ? examples/s]

In [47]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")
confusion_metric = evaluate.load("confusion_matrix")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=preds, references=labels)
    precision = precision_metric.compute(predictions=preds, references=labels, average="macro")
    recall = recall_metric.compute(predictions=preds, references=labels, average="macro")
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")

    mcc = matthews_corrcoef(labels, preds)
    bal_acc = balanced_accuracy_score(labels, preds)
    
    cr = classification_report(labels, preds, target_names=train_dataset.features[LABEL_COLUMN].names)
    cm = confusion_metric.compute(predictions=preds, references=labels)

    print("Confusion Matrix:\n", cm["confusion_matrix"])
    print("Classification Report:\n", cr)
    print("MCC:", mcc)
    print("Balanced Accuracy:", bal_acc)

    return {
        "accuracy": accuracy["accuracy"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "f1": f1["f1"],
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
    }

In [48]:
# Class weight balancing
class_weights = torch.tensor(
    compute_class_weight(
        "balanced", 
        classes=np.unique(train_dataset[LABEL_COLUMN]), 
        y=train_dataset[LABEL_COLUMN]
    ), dtype=torch.float
)

In [49]:
# model = LettersBERTModule(model_name=MODEL_NAME, num_labels=len(LABELS), class_weights=class_weights)

In [50]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=len(train_dataset.features[LABEL_COLUMN].names)
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [51]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [52]:
# Training arguments
training_args = TrainingArguments(
    output_dir=f"./{OUTPUT_NAME}",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=50,
    no_cuda=not torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

/home/hice1/mwesley32/.local/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_520114/182404050.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [53]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mcc,Balanced Accuracy
1,0.515200,0.485844,0.786910,0.885246,0.625668,0.636040,0.440060,0.625668
2,0.483100,0.452121,0.789954,0.846423,0.639066,0.654905,0.438979,0.639066


Confusion Matrix:
 [[470   0]
 [140  47]]
Classification Report:
               precision    recall  f1-score   support

        male       0.77      1.00      0.87       470
      female       1.00      0.25      0.40       187

    accuracy                           0.79       657
   macro avg       0.89      0.63      0.64       657
weighted avg       0.84      0.79      0.74       657

MCC: 0.44006024596115506
Balanced Accuracy: 0.625668449197861
Confusion Matrix:
 [[465   5]
 [133  54]]
Classification Report:
               precision    recall  f1-score   support

        male       0.78      0.99      0.87       470
      female       0.92      0.29      0.44       187

    accuracy                           0.79       657
   macro avg       0.85      0.64      0.65       657
weighted avg       0.82      0.79      0.75       657

MCC: 0.4389789663510566
Balanced Accuracy: 0.6390658778017977


TrainOutput(global_step=330, training_loss=0.5199920163010106, metrics={'train_runtime': 101.4445, 'train_samples_per_second': 51.812, 'train_steps_per_second': 3.253, 'total_flos': 696248647335936.0, 'train_loss': 0.5199920163010106, 'epoch': 2.0})

In [54]:
trainer.evaluate()

Confusion Matrix:
 [[465   5]
 [133  54]]
Classification Report:
               precision    recall  f1-score   support

        male       0.78      0.99      0.87       470
      female       0.92      0.29      0.44       187

    accuracy                           0.79       657
   macro avg       0.85      0.64      0.65       657
weighted avg       0.82      0.79      0.75       657

MCC: 0.4389789663510566
Balanced Accuracy: 0.6390658778017977


{'eval_loss': 0.4521210193634033,
 'eval_accuracy': 0.7899543378995434,
 'eval_precision': 0.8464231052661414,
 'eval_recall': 0.6390658778017977,
 'eval_f1': 0.6549054535489175,
 'eval_mcc': 0.4389789663510566,
 'eval_balanced_accuracy': 0.6390658778017977,
 'eval_runtime': 3.8817,
 'eval_samples_per_second': 169.254,
 'eval_steps_per_second': 10.82,
 'epoch': 2.0}